In [11]:
import numpy as np
import random as rd

theta = 2

def p(x: float, th: float) -> float:
    return (th - 1) / (x ** th) if x >= 1 else 0

def F(x: float, th: float) -> float:
    return 1 - x ** (1 - th) if x >= 1 else 0

def F_inv(y: float, th: float) -> float:
    return (1 - y) ** (1 / (1 - th))

# Размер выборки
N = 100
arr = []

for _ in range(N):
    y = rd.random()
    arr.append(F_inv(y, theta))

arr = np.array(arr)

# Доверительный интервал для медианы
beta = 0.95
mean_ln = np.mean([np.log(x) for x in arr])

g_theta = 2 ** (mean_ln)

lower_bound_median = g_theta - 1.96 * g_theta * np.log(2) * mean_ln / np.sqrt(N)
upper_bound_median = g_theta + 1.96 * g_theta * np.log(2) * mean_ln / np.sqrt(N)

print("Доверительный интервал для медианы:")
print(f"({lower_bound_median}, {upper_bound_median}), длина: {upper_bound_median - lower_bound_median}")

variation_row = sorted(arr)
print("Медиана:", (variation_row[49] + variation_row[50]) / 2)
print()

# Асимптотический доверительный интервал для theta
lower_bound_asymp = 1 - (1.96 - np.sqrt(N)) / (mean_ln * np.sqrt(N))
upper_bound_asymp = 1 + (1.96 + np.sqrt(N)) / (mean_ln * np.sqrt(N))

print("Асимптотический доверительный интервал для theta:")
print(f"({lower_bound_asymp}, {upper_bound_asymp}), длина: {upper_bound_asymp - lower_bound_asymp}")
print()

# Непараметрический бутстрап для theta
n_iterations = 1000
beta = 0.95

h_wave = 1 + 1 / mean_ln
bootstrap_delta = []

for _ in range(n_iterations):
    sample = np.random.choice(arr, size=N, replace=True)
    bootstrap_delta.append(1 + 1 / np.mean([np.log(x) for x in sample]) - h_wave)

variation_row = sorted(bootstrap_delta)

t_1 = variation_row[int((1 - beta) * n_iterations / 2)]
t_2 = variation_row[int((1 + beta) * n_iterations / 2)]

lower_bound_nonparam = h_wave - t_2
upper_bound_nonparam = h_wave - t_1

print("Непараметрический бутстраповский доверительный интервал для theta:")
print(f"({lower_bound_nonparam}, {upper_bound_nonparam}), длина: {upper_bound_nonparam - lower_bound_nonparam}")
print()

# Параметрический бутстрап для theta
n_iterations = 50000
beta = 0.95

h_wave = 1 + 1 / mean_ln
bootstrap_delta = []

for _ in range(n_iterations):
    sample = []
    for _ in range(N):
        y = rd.random()
        sample.append(F_inv(y, h_wave))

    bootstrap_delta.append(1 + 1 / np.mean([np.log(x) for x in sample]) - h_wave)

variation_row = sorted(bootstrap_delta)

t_1 = variation_row[int((1 - beta) * n_iterations / 2)]
t_2 = variation_row[int((1 + beta) * n_iterations / 2)]

lower_bound_param = h_wave - t_2
upper_bound_param = h_wave - t_1

print("Параметрический бутстраповский доверительный интервал для theta:")
print(f"({lower_bound_param}, {upper_bound_param}), длина: {upper_bound_param - lower_bound_param}")
print()

# Сравнение интервалов для theta
theta_intervals = {
    "непараметрического бутстраповского": upper_bound_nonparam - lower_bound_nonparam,
    "асимптотического": upper_bound_asymp - lower_bound_asymp,
    "параметрического бутстраповского": upper_bound_param - lower_bound_param
}

ordered = sorted(theta_intervals.items(), key=lambda x: x[1])

comparison_text = " < ".join(
    [f"Длина {name} дов. инт." for name, _ in ordered]
)

print("Сравнение интервалов:")
print(comparison_text)

Доверительный интервал для медианы:
(1.763522241398824, 2.342359481914959), длина: 0.578837240516135
Медиана: 1.9184127301389666

Асимптотический доверительный интервал для theta:
(1.7747963236341184, 2.1525577152567235), длина: 0.377761391622605

Непараметрический бутстраповский доверительный интервал для theta:
(1.7054830239344199, 2.1333940883644207), длина: 0.4279110644300008

Параметрический бутстраповский доверительный интервал для theta:
(1.741765280955386, 2.1282156813621422), длина: 0.38645040040675616

Сравнение интервалов:
Длина асимптотического дов. инт. < Длина параметрического бутстраповского дов. инт. < Длина непараметрического бутстраповского дов. инт.
